In [1]:
import numpy as np
import pandas as pd

from md_Helpers import (
    ThermalizationConfig,
    run_thermalization,
)


# ============================================================
# Scan ranges — endpoints are included
# ============================================================

kT_values = np.round(
    np.arange(0.7, 1.0 + 0.01, 0.02),
    decimals=10,
)

rho_values = np.round(
    np.arange(0.54, 0.6 + 0.01, 0.02),
    decimals=10,
)


# ============================================================
# Core thermalization settings
# ============================================================

n_cells = 40
nsteps = 200_000
dt = 0.002

# HDF5 log interval in simulation steps.
nlog = 1_000

seed = 1



# ============================================================
# Execution controls
# ============================================================

base_notes = (
    "Thermalization grid scan over temperature and density"
)

# If True, immediately stop when any grid point fails.
# If False, record the failure and continue through the grid.
stop_on_error = False


# ============================================================
# Validate the requested grid before starting
# ============================================================

if len(kT_values) == 0:
    raise ValueError("kT_values cannot be empty.")

if len(rho_values) == 0:
    raise ValueError("rho_values cannot be empty.")

if np.any(kT_values <= 0):
    raise ValueError("Every kT value must be positive.")

if np.any(rho_values <= 0):
    raise ValueError("Every density must be positive.")

if n_cells <= 0:
    raise ValueError("n_cells must be positive.")

if nsteps <= 0:
    raise ValueError("nsteps must be positive.")

if nlog <= 0:
    raise ValueError("nlog must be positive.")

if dt <= 0:
    raise ValueError("dt must be positive.")

# The workflow requires at least 41 evolved log points for
# its five terminal phase-analysis frames.
number_of_evolved_logs = int(np.ceil(nsteps / nlog))

if number_of_evolved_logs < 41:
    raise ValueError(
        "This workflow requires at least 41 evolved log points. "
        f"The current settings provide only "
        f"{number_of_evolved_logs}. Increase nsteps or reduce nlog."
    )

number_of_runs = len(kT_values) * len(rho_values)
number_of_particles = 4 * n_cells**3

print("Thermalization scan")
print(f"  kT values:       {kT_values.tolist()}")
print(f"  density values:  {rho_values.tolist()}")
print(f"  grid size:       {number_of_runs} runs")
print(f"  particles/run:   {number_of_particles:,}")
print(f"  steps/run:       {nsteps:,}")
print(f"  dt:              {dt:g}")
print(f"  log period:      {nlog:,}")
print()


# ============================================================
# Run every kT-density combination
# ============================================================

results = []
scan_index = 0

for kT in kT_values:
    for rho in rho_values:
        scan_index += 1

        kT = float(kT)
        rho = float(rho)

        print(
            f"[{scan_index:>2}/{number_of_runs}] "
            f"Starting kT={kT:.3f}, rho={rho:.3f}"
        )

        config = ThermalizationConfig(
            n_fcc_cells=n_cells,
            target_rho=rho,
            nsteps=nsteps,
            kT=kT,
            dt=dt,
            log_period=nlog,
            seed=seed,
        )

        try:
            result = run_thermalization(config)

            created_new = bool(
                result.get("created_new", False)
            )
            skipped = bool(result.get("skipped", False))

            if created_new:
                action = "created"
            elif skipped:
                action = "reused"
            else:
                action = "returned"

            print(
                f"    {action}: "
                f"Run_ID={result['run_id']}, "
                f"status={result.get('status', 'unknown')}"
            )

            results.append({
                "scan_index": scan_index,
                "kT": kT,
                "rho": rho,
                "run_id": result["run_id"],
                "action": action,
                "status": result.get("status"),
                "created_new": created_new,
                "skipped_existing": skipped,
                "run_signature": result.get(
                    "run_signature"
                ),
                "error": None,
            })

        except Exception as error:
            print(
                f"    FAILED: "
                f"{type(error).__name__}: {error}"
            )

            results.append({
                "scan_index": scan_index,
                "kT": kT,
                "rho": rho,
                "run_id": None,
                "action": "failed",
                "status": "Failed",
                "created_new": False,
                "skipped_existing": False,
                "run_signature": config.run_signature,
                "error": (
                    f"{type(error).__name__}: {error}"
                ),
            })

            if stop_on_error:
                raise


# ============================================================
# Display the completed scan
# ============================================================

thermalization_scan = (
    pd.DataFrame(results)
    .sort_values(["kT", "rho"])
    .reset_index(drop=True)
)

print()
print("Thermalization scan results:")
display(thermalization_scan)

print()
print("Run-ID grid:")

run_id_grid = thermalization_scan.pivot(
    index="kT",
    columns="rho",
    values="run_id",
)

display(run_id_grid)

failed_runs = thermalization_scan.loc[
    thermalization_scan["action"] == "failed"
]

if failed_runs.empty:
    print(
        f"All {number_of_runs} grid points completed "
        "or reused an existing matching run."
    )
else:
    print(
        f"{len(failed_runs)} of {number_of_runs} "
        "grid points failed:"
    )
    display(
        failed_runs[
            ["kT", "rho", "error"]
        ]
    )

Thermalization scan
  kT values:       [0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
  density values:  [0.54, 0.56, 0.58, 0.6]
  grid size:       64 runs
  particles/run:   256,000
  steps/run:       200,000
  dt:              0.002
  log period:      1,000

[ 1/64] Starting kT=0.700, rho=0.540
    reused: Run_ID=20260916183629, status=Cancelled
[ 2/64] Starting kT=0.700, rho=0.560
    reused: Run_ID=20260916183643, status=Cancelled
[ 3/64] Starting kT=0.700, rho=0.580
    reused: Run_ID=20260916183820, status=Running
[ 4/64] Starting kT=0.700, rho=0.600
    created: Run_ID=20260916183907, status=Complete
[ 5/64] Starting kT=0.720, rho=0.540
    reused: Run_ID=20260916184243, status=Complete
[ 6/64] Starting kT=0.720, rho=0.560
    reused: Run_ID=20260916184708, status=Running
[ 7/64] Starting kT=0.720, rho=0.580
    created: Run_ID=20260916184713, status=Complete
[ 8/64] Starting kT=0.720, rho=0.600
    reused: Run_ID=20260916185135, sta

,scan_index,kT,rho,run_id,action,status,created_new,skipped_existing,run_signature,error
0,1,0.70,0.54,20260916183629,reused,Cancelled,False,True,3766cf7d4a0622c94bc1d395cc86d5da508ae149aec06c...,None
1,2,0.70,0.56,20260916183643,reused,Cancelled,False,True,b8f405a21df19611b9cd09712376bfafa253d238aa8dfe...,None
2,3,0.70,0.58,20260916183820,reused,Running,False,True,207af281977e6d305e48e7ee4f38991dc87afb08d4b0f5...,None
3,4,0.70,0.60,20260916183907,created,Complete,True,False,a70f17dc1a08d30c6e12e9f228692a8a0b5e8ff7278f17...,None
4,5,0.72,0.54,20260916184243,reused,Complete,False,True,a9a97835547136d11bb95b35837eef073098b574826f7e...,None
...,...,...,...,...,...,...,...,...,...,...
59,60,0.98,0.60,20260916211642,reused,Complete,False,True,e65adc6307f53d9cd162325526a25479478c7bcda40dcb...,None
60,61,1.00,0.54,20260916212047,reused,Running,False,True,50fa32c67db43bf43bea4c9db343ca0ad37a76f8bb8d48...,None
61,62,1.00,0.56,20260916212312,created,Complete,True,False,80f4daa218fa50ecc431033507f0c323c927ac31403664...,None
62,63,1.00,0.58,20260916212451,reused,Complete,False,True,ee155188c550185e8b6de5b7dc69add3f7f84af3dc6bfb...,None



Run-ID grid:


rho,0.54,0.56,0.58,0.60
kT,,,,
0.70,20260916183629,20260916183643,20260916183820,20260916183907
0.72,20260916184243,20260916184708,20260916184713,20260916185135
0.74,20260916185510,20260916185600,20260916190022,20260916190310
0.76,20260916190438,20260916190854,20260916191216,20260916191410
0.78,20260916191833,20260916192009,20260916192301,20260916192710
0.80,20260916192808,20260916193125,20260916193534,20260916193557
0.82,20260916194103,20260916194504,20260916194505,20260916194905
0.84,20260916195244,20260916195312,20260916195726,20260916200033
0.86,20260916200204,20260916200654,20260916200832,20260916201103


All 64 grid points completed or reused an existing matching run.
